In [ ]:
# Shared repository paths (no model or dataset is loaded here).
from pathlib import Path as _RepositoryPath
import os as _repository_os
import sys as _repository_sys

_repository_hint = _repository_os.environ.get("TESI_REPOSITORY_ROOT")
_repository_starts = (
    [_RepositoryPath(_repository_hint).expanduser().resolve()]
    if _repository_hint else
    ([_RepositoryPath(__file__).resolve().parent] if "__file__" in globals() else [])
    + [_RepositoryPath.cwd().resolve()]
)
_repository_root = next(
    (candidate for start in _repository_starts for candidate in (start, *start.parents)
     if (candidate / "project_paths.py").is_file()), None
)
if _repository_root is None:
    raise FileNotFoundError(
        "Start Jupyter inside the repository or set TESI_REPOSITORY_ROOT to its root."
    )
if str(_repository_root) not in _repository_sys.path:
    _repository_sys.path.insert(0, str(_repository_root))
from project_paths import PROJECT_ROOT as TESI_PROJECT_ROOT
from project_paths import BASELINES_PATH as TESI_BASELINES_PATH


# RouterPolicyAnalysis — risultati leggibili per seed e per task
Il report contiene la stessa analisi per ciascun seed, il riepilogo con media, deviazione standard e bootstrap, i confronti con baseline separati per SST e MNLI, e i grafici delle sole configurazioni routed.
Misure, cache e percorso di salvataggio restano invariati. Le quattro sezioni iniziali sono richiudibili.


In [ ]:
from pathlib import Path
from datetime import datetime
from html import escape
import json
import math
import re

# Un punto percentuale F1; risparmio FLOPs strettamente superiore all'1%.
# Esempio: F1 90% -> 89% rientra nella tolleranza.
QUALITY_TOLERANCE_PP = 1.0
MIN_FLOPS_SAVING_PERCENT = 1.0

# Normalmente non serve impostarlo: avviare da Baselines o una sua sottocartella.
BASELINES_OVERRIDE = TESI_BASELINES_PATH  # Shared configured baseline directory.

if not (math.isfinite(QUALITY_TOLERANCE_PP) and 0 <= QUALITY_TOLERANCE_PP <= 100):
    raise ValueError("La tolleranza F1 deve essere tra 0 e 100 punti percentuali.")
if not (math.isfinite(MIN_FLOPS_SAVING_PERCENT) and 0 <= MIN_FLOPS_SAVING_PERCENT < 100):
    raise ValueError("La soglia di risparmio deve essere tra 0 e 100%, escluso 100.")

def find_baselines():
    if BASELINES_OVERRIDE is not None:
        path = Path(BASELINES_OVERRIDE).expanduser().resolve()
        if not (path / "RoutedModel" / "Results").is_dir():
            raise FileNotFoundError(f"Manca RoutedModel/Results in {path}")
        return path
    cwd = Path.cwd().resolve()
    for parent in (cwd, *cwd.parents):
        for candidate in (parent, parent / "Baselines",
                          parent / "Secondo_Esperimento" / "Baselines"):
            if (candidate / "RoutedModel" / "Results").is_dir():
                return candidate
    raise FileNotFoundError("Avviare da Baselines oppure impostare BASELINES_OVERRIDE.")

BASELINES_PATH = find_baselines()
RUNS_ROOT = BASELINES_PATH / "RoutedModel" / "Results"
CHOICES = ("SST", "MNLI", "MERGED", "IDENTITY")
TASKS = ("sst", "mnli")
NOTES = []
SOURCES = []

def read_json(path):
    with Path(path).open(encoding="utf-8-sig") as handle:
        return json.load(handle)

def latest_completed_run(root):
    folders = sorted(
        (p for p in root.iterdir()
         if p.is_dir() and re.fullmatch(r"\d{8}_\d{6}(?:_\d{6})?", p.name)),
        key=lambda p: p.name, reverse=True)
    required = {f"{t}_{s}" for t in TASKS for s in ("validation", "test")}
    skipped = []
    for parent in folders:
        children = sorted(parent.glob(f"{parent.name}_seed_*"))
        candidates = children if children else [parent]
        completed = []
        for folder in candidates:
            path = folder / "comparison_manifest.json"
            try:
                item = read_json(path)
            except (ValueError, OSError):
                skipped.append(folder.name + ": manifest assente o non leggibile")
                continue
            if (item.get("status") != "evaluation_completed"
                    or not required.issubset(item.get("splits", {}))
                    or (children and item.get("measurement_status") != "completed")):
                skipped.append(folder.name + ": valutazione/misure incomplete")
                continue
            completed.append((folder, item))
        if completed:
            NOTES.extend("Run ignorata: " + reason for reason in skipped)
            for folder, _ in completed:
                SOURCES.append(("Run routed: metriche, routing e misure", str(folder / "comparison_manifest.json")))
            return parent, completed
    raise FileNotFoundError("Nessuna run completata. " + "; ".join(skipped))



GROUP_DIR, GROUP_RUNS = latest_completed_run(RUNS_ROOT)
RUN_DIR, manifest = GROUP_RUNS[0]
OUTPUT_PATH = GROUP_DIR / "routing_policy_report.html"
print("Esperimento:", GROUP_DIR.name, "| seed completati:", len(GROUP_RUNS))
print("Dettaglio routing principale:", RUN_DIR.name, "(primo completato, non selezionato sul test)")
print("Report HTML:", OUTPUT_PATH)


## Lettura e confronti

Le F1 sono ricalcolate dalle matrici di confusione con una convenzione unica:
binaria (classe 1) per SST, macro per MNLI. Non si mescolano validation e test.
Se mancano report o costi confrontabili, il report lo segnala invece di inventare valori.


In [ ]:
def metrics_from_result(result, task):
    metrics = result.get("metrics", result)
    matrix = result.get("confusion_matrix", metrics.get("confusion_matrix"))
    k = 2 if task == "sst" else 3
    if not isinstance(matrix, list) or len(matrix) != k or any(len(row) != k for row in matrix):
        raise ValueError(f"{task}: matrice di confusione assente o non valida.")
    if any(not isinstance(v, (int, float)) or not math.isfinite(v) or v < 0 or int(v) != v
           for row in matrix for v in row):
        raise ValueError(f"{task}: conteggi della matrice non validi.")
    support = [sum(row) for row in matrix]
    predicted = [sum(matrix[i][j] for i in range(k)) for j in range(k)]
    n = sum(support)
    if n <= 0:
        raise ValueError(f"{task}: nessun esempio.")
    declared_n = result.get("num_samples", metrics.get("num_samples"))
    if declared_n is not None and declared_n != n:
        raise ValueError(f"{task}: numero campioni diverso dalla matrice.")
    f1s = [2 * matrix[i][i] / (support[i] + predicted[i])
           if support[i] + predicted[i] else 0.0 for i in range(k)]
    return {"n": int(n), "support": support,
            "accuracy": sum(matrix[i][i] for i in range(k)) / n,
            "f1": f1s[1] if task == "sst" else sum(f1s) / k}

def measured_flops(cost):
    coverage = cost.get("coverage", {})
    value = cost.get("flops_per_example")
    if coverage.get("status") == "complete_for_convention" and isinstance(value, (int, float)):
        if math.isfinite(value) and value > 0:
            return float(value), ""
    missing = set(coverage.get("uncovered_operators", []))
    counted, n = coverage.get("counted_flops"), cost.get("num_examples")
    # Padding costante copia/riempie valori: non aggiunge FLOPs aritmetici.
    if (missing == {"aten.constant_pad_nd"} and isinstance(counted, (int, float))
            and math.isfinite(counted) and counted > 0
            and isinstance(n, (int, float)) and math.isfinite(n) and n > 0):
        return counted / n, "FLOPs recuperati dal conteggio: unica operazione esclusa, padding costante."
    return None, "FLOPs mancanti o conteggio incompleto."

def make_entry(name, task, result, primary=False, params=None):
    cost = result.get("inference_cost", result.get("metrics", {}).get("inference_cost", {}))
    flops, note = measured_flops(cost)
    return {"name": name, "task": task, "primary": primary,
            "metrics": metrics_from_result(result, task),
            "cost": cost, "flops": flops, "cost_note": note, "params": params}

ROUTED = {
    task: make_entry("Routed", task, manifest["splits"][f"{task}_test"],
                     params=manifest.get("experiment", {}).get("trainable_params"))
    for task in TASKS
}
BASELINE_FILES = (
    ("sst_expert", "Esperto SST", "EvaluationExpertSST/Results/expert_a_metrics.json"),
    ("mnli_expert", "Esperto MNLI", "EvaluationExpertMNLI/Results/expert_b_metrics.json"),
    ("weight_averaging", "Weight averaging",
     "EvaluationExpertWeigthAveraging/Results/weight_averaging_test_results.json"),
    ("multitask", "Multitask BERT",
     "EvaluationExpertMultitaskBert/Results/multitask_bert_test_results.json"),
)
BASELINES = []
BASELINE_REPORTS = []
for key, label, relative in BASELINE_FILES:
    task = "sst" if key == "sst_expert" else "mnli" if key == "mnli_expert" else None
    measured_reference = manifest.get("fixed_expert_test", {}).get(task)
    if measured_reference is not None:
        # Prefer the exact fixed expert evaluated alongside this routed experiment.
        BASELINES.append(make_entry(label + " (del task)", task, measured_reference, True, 0))
        SOURCES.append((label + " (stessi esempi del routed)", str(RUN_DIR / "comparison_manifest.json")))
        continue
    path = BASELINES_PATH / relative
    source_label = label + " (report corrente)"
    if not path.is_file():
        # Le copie appartengono a questa run; nessuna dipendenza dai vecchi percorsi assoluti.
        path = RUN_DIR / "baseline_reports" / f"{key}.json"
        source_label = label + " (copia salvata nella run)"
    if not path.is_file():
        NOTES.append(label + ": report assente; confronto non disponibile.")
        continue
    SOURCES.append((source_label, str(path)))
    report = read_json(path)
    BASELINE_REPORTS.append((label, report))
    if "methods" in report:
        for method in report["methods"]:
            task = method.get("eval_task")
            if task not in TASKS:
                continue
            result = method.get("results", {}).get(task, {}).get("test_metrics")
            if result is None:
                NOTES.append(f"{label} su {task.upper()}: metriche test assenti.")
                continue
            in_domain = method.get("mode") == "in_domain"
            name = label + (" (del task)" if in_domain else " (cross-task)")
            BASELINES.append(make_entry(name, task, result, in_domain,
                                        method.get("trainable_params")))
    else:
        for task in TASKS:
            if task not in report:
                NOTES.append(f"{label} su {task.upper()}: metriche test assenti.")
                continue
            BASELINES.append(make_entry(label, task, report[task],
                                        params=report.get("trainable_params")))

def compare(routed, baseline):
    rm, bm = routed["metrics"], baseline["metrics"]
    same_samples = rm["n"] == bm["n"] and rm["support"] == bm["support"]
    delta = 100 * (rm["f1"] - bm["f1"]) if same_samples else None
    rc, bc = routed["cost"], baseline["cost"]
    ri, bi = rc.get("input", {}), bc.get("input", {})
    lengths_r = sorted(ri.get("observed_sequence_lengths", []))
    lengths_b = sorted(bi.get("observed_sequence_lengths", []))
    convention_r, convention_b = rc.get("convention", {}), bc.get("convention", {})
    same_cost_protocol = (
        bool(lengths_r) and lengths_r == lengths_b
        and all(convention_r.get(key) is not None
                and convention_r.get(key) == convention_b.get(key)
                for key in ("multiply_accumulate_flops",
                            "elementwise_arithmetic_or_special_function_flops",
                            "gelu_flops_per_element", "layer_norm", "softmax", "attention", "excluded"))
        and ri.get("padding") == bi.get("padding")
        and ri.get("truncation") == bi.get("truncation")
    )
    saving = None
    if (same_samples and same_cost_protocol
            and routed["flops"] is not None and baseline["flops"] is not None):
        saving = 100 * (1 - routed["flops"] / baseline["flops"])
    if delta is None:
        quality = "Non confrontabile: numerosità o distribuzione delle classi diversa"
    elif delta < -QUALITY_TOLERANCE_PP - 1e-9:
        quality = "Sotto la tolleranza"
    elif delta < -1e-9:
        quality = "Entro la tolleranza"
    elif delta > 1e-9:
        quality = "F1 maggiore nel campione"
    else:
        quality = "Stessa F1"
    if not same_samples:
        reason = "Campioni/classi non allineati"
    elif not same_cost_protocol:
        reason = "Configurazioni o convenzioni FLOPs non confrontabili"
    elif saving is None:
        reason = "Conteggio FLOPs non disponibile"
    else:
        reason = ""
    return {"delta": delta, "saving": saving, "quality": quality, "cost_reason": reason}

COMPARISONS = [(entry, compare(ROUTED[entry["task"]], entry)) for entry in BASELINES]


# Costi delle sole procedure sperimentali: nessun costo originario degli esperti.
COST_NOTES = []

def valid_cost(value):
    return (isinstance(value, (int, float)) and not isinstance(value, bool)
            and math.isfinite(value) and value >= 0)

def training_record(name, task, cost=None, epochs=None, history=None,
                    scope="", evaluation_only=False):
    cost = cost if isinstance(cost, dict) else {}
    history = history if isinstance(history, list) else []
    per_epoch = cost.get("per_epoch")
    if not isinstance(per_epoch, list):
        per_epoch = [
            dict(row["training_cost"], epoch=row.get("epoch", index))
            for index, row in enumerate(history, 1)
            if isinstance(row, dict) and isinstance(row.get("training_cost"), dict)
        ]
    if epochs is None:
        epochs = cost.get("epochs_completed")
    if epochs is None and history:
        epochs = len(history)
    if epochs is None and per_epoch:
        # Use the last declared epoch, not the number of potentially partial records.
        numbers = [row.get("epoch") for row in per_epoch if isinstance(row, dict)]
        if len(numbers) == len(per_epoch) and all(type(n) is int and n > 0 for n in numbers):
            epochs = max(numbers)
    if type(epochs) is not int or epochs < 0:
        epochs = None

    if evaluation_only:
        return {"name": name, "task": task, "epochs": 0, "train": 0.0,
                "validation": 0.0, "total": 0.0, "mean": None,
                "per_epoch": [], "preparation": None,
                "scope": "Solo valutazione: nessun training aggiuntivo."}

    train = cost.get("training_flops")
    validation = cost.get("validation_flops")
    total = cost.get("total_flops")
    train = train if valid_cost(train) else None
    validation = validation if valid_cost(validation) else None
    total = total if valid_cost(total) else None

    # Same arithmetic convention as inference: padding is copying, not arithmetic.
    coverage = cost.get("coverage") or {}
    missing = set(coverage.get("uncovered_operators", []))
    counted = coverage.get("counted_flops")
    if (total is None and missing == {"aten.constant_pad_nd"}
            and valid_cost(counted)):
        total = counted
        if validation is None and train is not None and total >= train:
            validation = total - train
        COST_NOTES.append(
            name + ": totale recuperato dal conteggio; l'unica operazione esclusa "
            "è il padding. La validation, se assente, è totale meno training.")
    if total is None and train is not None and validation is not None:
        total = train + validation
    preparation = (cost.get("preparation_cost") or {}).get("total_flops")
    preparation = preparation if valid_cost(preparation) else None
    if not cost:
        scope += " Conteggio di training non presente nei risultati disponibili."
    return {"name": name, "task": task, "epochs": epochs, "train": train,
            "validation": validation, "total": total,
            "mean": train / epochs if train is not None and epochs else None,
            "per_epoch": per_epoch, "preparation": preparation, "scope": scope}

TRAINING = []
summary_path = RUN_DIR / "routed_training_summary.json"
run_summary = {}
if summary_path.is_file():
    try:
        run_summary = read_json(summary_path)
        if not isinstance(run_summary, dict):
            raise ValueError("Summary non valido")
        SOURCES.append(("Routed: costi di training", str(summary_path)))
    except (ValueError, OSError) as error:
        COST_NOTES.append("Summary di training non leggibile: " + str(error))
        run_summary = {}
else:
    COST_NOTES.append("Summary di training routed assente.")
TRAINING.append(training_record(
    "Routed", "SST + MNLI", run_summary.get("training_cost"),
    run_summary.get("epochs_completed"),
    scope="Training di router e teste; tutti gli esperti congelati."))

for label, report in BASELINE_REPORTS:
    if "methods" in report:
        for method in report["methods"]:
            task = method.get("eval_task")
            if task not in TASKS:
                continue
            in_domain = method.get("mode") == "in_domain"
            name = label + (" (del task)" if in_domain else " (cross-task)")
            result = method.get("results", {}).get(task, {})
            best_summary = result.get("best_summary") or {}
            cost = result.get("training_cost") or best_summary.get("training_cost")
            TRAINING.append(training_record(
                name, task.upper(), cost, best_summary.get("epochs_completed"),
                result.get("train_history"),
                scope="Adattamento della testa al task; backbone congelato.",
                evaluation_only=in_domain))
    else:
        scope = ("Training delle teste sul backbone merged congelato."
                 if label == "Weight averaging"
                 else "Training congiunto del modello sui due task.")
        TRAINING.append(training_record(
            label, "SST + MNLI", report.get("training_cost"),
            report.get("epochs_completed"), scope=scope))


## Routing e skipping già osservati

Le percentuali per layer sono quelle salvate dal modello sulla validation.
La tabella riporta anche i layer attivi sul test, per affiancarli alle metriche di test.
Una scelta frequente dello stesso esperto non è automaticamente un errore.


In [ ]:
def routing_summary(task, split):
    record = manifest["splits"][f"{task}_{split}"]
    routing = record.get("routing")
    if not isinstance(routing, dict):
        raise ValueError(f"Routing mancante: {task}/{split}")
    n = metrics_from_result(record, task)["n"]
    layers = routing.get("num_layers")
    if not isinstance(layers, int) or layers <= 0 or routing.get("num_examples") != n:
        raise ValueError(f"Routing non allineato alle metriche: {task}/{split}")
    per_layer = routing.get("per_layer", [])
    if len(per_layer) != layers or [r.get("layer") for r in per_layer] != list(range(1, layers + 1)):
        raise ValueError(f"Layer mancanti o disordinati: {task}/{split}")
    for row in per_layer:
        percentages = row.get("hard_choice_percentages", {})
        if set(percentages) != set(CHOICES):
            raise ValueError(f"Scelte routing mancanti: {task}/{split}")
        if any(not isinstance(v, (int, float)) or not math.isfinite(v)
               or not 0 <= v <= 100 for v in percentages.values()):
            raise ValueError(f"Percentuali routing non valide: {task}/{split}")
        if not math.isclose(sum(percentages.values()), 100, abs_tol=0.001):
            raise ValueError(f"Percentuali routing non sommano a 100: {task}/{split}")
    active = sum(1 - row["hard_choice_percentages"]["IDENTITY"] / 100 for row in per_layer)
    declared = routing.get("average_active_non_identity_layers_per_example")
    if declared is not None and not math.isclose(active, declared, abs_tol=0.001):
        raise ValueError(f"Media layer incoerente: {task}/{split}")
    shares = {c: sum(row["hard_choice_percentages"][c] for row in per_layer) / layers for c in CHOICES}
    return {"n": n, "layers": layers, "active": active, "skipped": layers - active,
            "shares": shares, "per_layer": per_layer,
            "unique_routes": routing.get("num_unique_routes")}

ROUTING = {(task, split): routing_summary(task, split)
           for task in TASKS for split in ("validation", "test")}

def judgments(task):
    primary = [(e, c) for e, c in COMPARISONS if e["task"] == task and e["primary"]]
    if len(primary) != 1:
        return ("Non determinabile: manca un riferimento univoco dell'esperto del task.",
                "Layer saltati descritti sotto; confronto qualità/costo non determinabile.",
                "Comportamento descritto dal grafico; utilità complessiva non determinabile.")
    entry, comparison = primary[0]
    delta, saving = comparison["delta"], comparison["saving"]
    quality = comparison["quality"]
    if delta is not None:
        quality += f": {delta:+.3f} punti F1 rispetto a {entry['name']}."
    if delta is None:
        return quality, "Non determinabile: confronto dei campioni non allineato.", "Utilità non determinabile."
    within = delta >= -QUALITY_TOLERANCE_PP - 1e-9
    active = ROUTING[(task, "test")]["active"]
    layers = ROUTING[(task, "test")]["layers"]
    if saving is None:
        skip = f"{active:.2f}/{layers} layer attivi; risparmio FLOPs non verificabile."
        policy = "Qualità descritta sopra; vantaggio di efficienza non verificabile."
    elif not within:
        skip = f"Risparmio FLOPs {saving:+.2f}%, ma qualità sotto la tolleranza."
        policy = "Da migliorare: il compromesso non rispetta la tolleranza di qualità."
    elif saving > MIN_FLOPS_SAVING_PERCENT + 1e-9:
        skip = f"Positivo: {saving:.2f}% di FLOPs risparmiati con qualità entro la tolleranza."
        policy = f"Compromesso utile: qualità entro la tolleranza e risparmio del {saving:.2f}%."
    elif saving > 0:
        skip = f"Risparmio contenuto ({saving:.2f}%): non supera la soglia scelta."
        policy = "Beneficio di efficienza contenuto; qualità entro la tolleranza."
    else:
        skip = f"Nessun risparmio FLOPs ({saving:+.2f}%), anche se alcuni layer possono essere saltati."
        policy = ("F1 maggiore nel campione, ma nessun vantaggio di efficienza."
                  if delta > 1e-9 else "Nessun vantaggio di qualità/costo dimostrato da questo confronto.")
    return quality, skip, policy


def overview_judgments(task):
    quality = judgments(task)[0]
    routing = ROUTING[(task, "test")]
    skip = (f"{routing['skipped']:.2f} layer saltati in media su {routing['layers']}; "
            f"{routing['shares']['IDENTITY']:.2f}% delle decisioni.")
    shares = routing["shares"]
    policy = (f"Scelte sul test: SST {shares['SST']:.2f}%, "
              f"MNLI {shares['MNLI']:.2f}%, merged {shares['MERGED']:.2f}%.")
    return quality, skip, policy


## Report HTML: seed, riepilogo statistico, baseline e configurazioni routed
Le misure completate vengono riutilizzate dall’archivio esistente. Cambia soltanto la presentazione del report.


In [ ]:
# Embedded definitions only: no routed training stage is executed.
INFERENCE_COMPONENT_SOURCE = 'import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom math import prod\nfrom torch.utils._pytree import tree_leaves\nfrom torch.utils.flop_counter import FlopCounterMode\n\n\ndef extract_layer_hidden_states(layer_output):\n    """\n        Handles both cases:\n        - layer_output is already a tensor: [batch, seq_len, hidden]\n        - layer_output is a tuple/list: first element is hidden states\n        """\n    if torch.is_tensor(layer_output):\n        return layer_output\n    if isinstance(layer_output, (tuple, list)):\n        return layer_output[0]\n    if hasattr(layer_output, \'last_hidden_state\'):\n        return layer_output.last_hidden_state\n    raise TypeError(f\'Unexpected layer output type: {type(layer_output)}\')\n\n\ndef expert_weight_identity(expert_path):\n    """Identify the actual frozen checkpoint, not only its directory name."""\n    from pathlib import Path\n    import hashlib\n\n    def file_sha256(path):\n        digest = hashlib.sha256()\n        with Path(path).open(\'rb\') as handle:\n            for block in iter(lambda: handle.read(1024 * 1024), b\'\'):\n                digest.update(block)\n        return digest.hexdigest()\n    expert_path = Path(expert_path)\n    files = [expert_path / \'classifier.pt\', expert_path / \'bert_backbone\' / \'config.json\']\n    files += sorted((expert_path / \'bert_backbone\').glob(\'*.safetensors\'))\n    files += sorted((expert_path / \'bert_backbone\').glob(\'pytorch_model*.bin\'))\n    if len(files) <= 2:\n        raise FileNotFoundError(f\'No backbone weight files in {expert_path}\')\n    return {str(path.relative_to(expert_path)): file_sha256(path) for path in files}\n\n\nclass ClassifierHead(nn.Module):\n\n    def __init__(self, input_dim=768, hidden_dim=256, num_classes=2, dropout_prob=0.1):\n        super().__init__()\n        self.classifier = nn.Sequential(nn.LayerNorm(input_dim), nn.Dropout(dropout_prob), nn.Linear(input_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout_prob), nn.Linear(hidden_dim, num_classes))\n\n    def forward(self, x):\n        return self.classifier(x)\n\n\nclass InferenceFlopCounter(FlopCounterMode):\n    """Extend PyTorch\'s matrix counts and expose uncovered floating-point ops."""\n    zero_flop_ops = set(\'\\n            alias detach view _unsafe_view reshape t transpose permute slice select\\n            squeeze unsqueeze expand expand_as as_strided contiguous clone copy_\\n            _to_copy to empty empty_like empty_strided new_empty new_empty_strided\\n            zeros zeros_like new_zeros ones ones_like full full_like new_full fill_\\n            zero_ embedding index index_select gather cat stack split split_with_sizes\\n            unbind masked_fill masked_fill_ where eq ne lt le gt ge argmax argmin\\n            max min amax amin any all isfinite isnan isinf scalar_tensor\\n            _local_scalar_dense lift_fresh lift_fresh_copy resolve_conj dropout\\n            native_dropout constant_pad_nd\\n        \'.split())\n\n    def __init__(self):\n        if not hasattr(FlopCounterMode, \'_count_flops\'):\n            raise RuntimeError(\'This FLOPs counter requires PyTorch with FlopCounterMode._count_flops support.\')\n        super().__init__(display=False)\n        self.uncovered_ops = set()\n\n    def _count_flops(self, packet, out, args, kwargs):\n        name = str(packet).split(\'.\')[-1]\n        outputs = [x for x in tree_leaves(out) if isinstance(x, torch.Tensor) and x.is_floating_point()]\n        if not outputs:\n            return out\n        n = outputs[0].numel()\n        x = args[0] if args else None\n        if name in {\'addmm\', \'baddbmm\'}:\n            alpha, beta = (kwargs.get(\'alpha\', 1), kwargs.get(\'beta\', 1))\n            flops = 2 * n * args[1].shape[-1]\n            flops += n * (int(alpha != 1) + int(beta != 0) + int(beta not in (0, 1)))\n        elif name in {\'add\', \'add_\', \'sub\', \'sub_\', \'rsub\'}:\n            flops = n * (1 + int(kwargs.get(\'alpha\', 1) != 1))\n        elif name in {\'mul\', \'mul_\', \'div\', \'div_\', \'true_divide\', \'neg\', \'exp\', \'log\', \'sqrt\', \'rsqrt\', \'reciprocal\', \'tanh\', \'erf\', \'pow\'}:\n            flops = n\n        elif name == \'gelu\':\n            flops = n * (9 if kwargs.get(\'approximate\', \'none\') == \'tanh\' else 5)\n        elif name in {\'native_layer_norm\', \'layer_norm\'}:\n            width = prod(args[1])\n            rows = x.numel() // width\n            weight = args[2] if len(args) > 2 else kwargs.get(\'weight\')\n            bias = args[3] if len(args) > 3 else kwargs.get(\'bias\')\n            flops = 5 * x.numel() + 2 * rows\n            flops += x.numel() * (int(weight is not None) + int(bias is not None))\n        elif name in {\'_softmax\', \'softmax\'}:\n            dim = args[1] if len(args) > 1 else kwargs[\'dim\']\n            flops = 4 * x.numel() - x.numel() // x.shape[dim]\n        elif name in {\'sum\', \'mean\'}:\n            flops = x.numel() - n + (n if name == \'mean\' else 0)\n        elif name in {\'scaled_dot_product_attention\', \'_scaled_dot_product_attention_math\', \'_scaled_dot_product_flash_attention\', \'_scaled_dot_product_flash_attention_for_cpu\', \'_scaled_dot_product_efficient_attention\', \'_scaled_dot_product_cudnn_attention\'}:\n            q, k, v = args[:3]\n            rows = prod(q.shape[:-1])\n            scores = rows * k.shape[-2]\n            mask = kwargs.get(\'attn_mask\', kwargs.get(\'attn_bias\'))\n            if mask is None and len(args) > 3 and isinstance(args[3], torch.Tensor):\n                mask = args[3]\n            flops = 2 * scores * (q.shape[-1] + v.shape[-1])\n            flops += 5 * scores - rows\n            if isinstance(mask, torch.Tensor) and mask.is_floating_point():\n                flops += scores\n        elif name in self.zero_flop_ops:\n            return out\n        elif packet in self.flop_registry:\n            if \'attention\' in name:\n                self.uncovered_ops.add(str(packet) + \' (attention overhead not covered)\')\n            return super()._count_flops(packet, out, args, kwargs)\n        else:\n            self.uncovered_ops.add(str(packet))\n            return out\n        self.flop_counts[\'Global\'][packet] += int(flops)\n        return out\n\n\nclass TrainingFlopCounter(InferenceFlopCounter):\n    """Count training tensor arithmetic without another forward/backward pass."""\n\n    def __init__(self, count_attention_saved_stats=False, embedding_padding_counts=None):\n        super().__init__()\n        self.count_attention_saved_stats = count_attention_saved_stats\n        self.embedding_padding_counts = embedding_padding_counts or {}\n    zero_flop_ops = InferenceFlopCounter.zero_flop_ops | {\'select_backward\', \'slice_backward\', \'diagonal_backward\', \'_reshape_alias\', \'as_strided_\', \'constant_pad_nd\', \'detach_\', \'unbind_backward\'}\n\n    def _count_flops(self, packet, out, args, kwargs):\n        name = str(packet).split(\'.\')[-1]\n        outputs = [x for x in tree_leaves(out) if isinstance(x, torch.Tensor) and x.is_floating_point()]\n        n = outputs[0].numel() if outputs else 0\n        if name == \'tanh_backward\':\n            self.flop_counts[\'Global\'][packet] += 3 * args[0].numel()\n            return out\n        if name in {\'embedding_dense_backward\', \'embedding_backward\'}:\n            grad, indices, num_weights, padding_idx, scale_by_frequency = args[:5]\n            sparse = args[5] if len(args) > 5 else False\n            if scale_by_frequency or sparse:\n                self.uncovered_ops.add(str(packet) + \' (frequency scaling/sparse gradient not covered)\')\n                return out\n            valid_rows = indices.numel()\n            if padding_idx >= 0:\n                key = (int(num_weights), int(padding_idx), int(indices.numel()))\n                if key not in self.embedding_padding_counts:\n                    self.uncovered_ops.add(str(packet) + \' (padding count unavailable)\')\n                    return out\n                valid_rows -= self.embedding_padding_counts[key]\n            self.flop_counts[\'Global\'][packet] += int(valid_rows * grad.shape[-1])\n            return out\n        if name in {\'index_add\', \'index_add_\', \'scatter_add\', \'scatter_add_\'}:\n            self.flop_counts[\'Global\'][packet] += args[3].numel() * (1 + int(kwargs.get(\'alpha\', 1) != 1))\n            return out\n        forward_attention = {\'scaled_dot_product_attention\', \'_scaled_dot_product_attention_math\', \'_scaled_dot_product_flash_attention\', \'_scaled_dot_product_flash_attention_for_cpu\', \'_scaled_dot_product_efficient_attention\', \'_scaled_dot_product_cudnn_attention\'}\n        backward_attention = {\'_scaled_dot_product_efficient_attention_backward\', \'_scaled_dot_product_flash_attention_backward\', \'_scaled_dot_product_flash_attention_for_cpu_backward\', \'_scaled_dot_product_cudnn_attention_backward\'}\n        if name in forward_attention or name in backward_attention:\n\n            def argument(index, key, default=None):\n                return args[index] if len(args) > index else kwargs.get(key, default)\n            is_backward = name in backward_attention\n            q, k, v = args[1:4] if is_backward else args[:3]\n            if q.ndim != 4 or k.ndim != 4 or v.ndim != 4 or (q.shape[:2] != k.shape[:2]) or (k.shape[:2] != v.shape[:2]):\n                self.uncovered_ops.add(str(packet) + \' (non-standard attention shape)\')\n                return out\n            mask = None\n            save_stats = False\n            if not is_backward:\n                if name in {\'_scaled_dot_product_efficient_attention\', \'_scaled_dot_product_cudnn_attention\'}:\n                    mask = argument(3, \'attn_bias\')\n                    save_stats = bool(argument(4, \'compute_log_sumexp\', False))\n                    dropout = argument(5, \'dropout_p\', 0.0)\n                    causal = argument(6, \'is_causal\', False)\n                elif name in {\'_scaled_dot_product_flash_attention\', \'_scaled_dot_product_flash_attention_for_cpu\'}:\n                    mask = kwargs.get(\'attn_mask\')\n                    dropout = argument(3, \'dropout_p\', 0.0)\n                    causal = argument(4, \'is_causal\', False)\n                    save_stats = self.count_attention_saved_stats\n                else:\n                    mask = argument(3, \'attn_mask\')\n                    dropout = argument(4, \'dropout_p\', 0.0)\n                    causal = argument(5, \'is_causal\', False)\n            elif name == \'_scaled_dot_product_efficient_attention_backward\':\n                mask = argument(4, \'attn_bias\')\n                dropout = argument(9, \'dropout_p\', 0.0)\n                grad_mask = argument(10, \'grad_input_mask\')\n                causal = argument(11, \'is_causal\', False)\n                if grad_mask is None or not all(grad_mask[:3]) or grad_mask[3]:\n                    self.uncovered_ops.add(str(packet) + \' (partial QKV or bias gradient not covered)\')\n                    return out\n            elif name == \'_scaled_dot_product_cudnn_attention_backward\':\n                mask = argument(8, \'attn_bias\')\n                dropout = argument(13, \'dropout_p\', 0.0)\n                causal = argument(14, \'is_causal\', False)\n            elif name == \'_scaled_dot_product_flash_attention_for_cpu_backward\':\n                mask = kwargs.get(\'attn_mask\')\n                dropout = argument(6, \'dropout_p\', 0.0)\n                causal = argument(7, \'is_causal\', False)\n            else:\n                dropout = argument(10, \'dropout_p\', 0.0)\n                causal = argument(11, \'is_causal\', False)\n            if causal or dropout < 0 or dropout >= 1:\n                self.uncovered_ops.add(str(packet) + \' (causal attention or unsupported dropout)\')\n                return out\n            rows = prod(q.shape[:-1])\n            scores = rows * k.shape[-2]\n            mask_additions = scores if isinstance(mask, torch.Tensor) and mask.is_floating_point() else 0\n            if not is_backward:\n                flops = 2 * scores * (q.shape[-1] + v.shape[-1])\n                flops += 5 * scores - rows + mask_additions\n                flops += 2 * scores if dropout > 0 else 0\n                flops += 2 * rows if save_stats and self.count_attention_saved_stats else 0\n            else:\n                flops = 2 * scores * (3 * q.shape[-1] + 2 * v.shape[-1])\n                flops += 6 * scores + mask_additions + (2 * args[0].numel() - rows)\n                flops += 4 * scores if dropout > 0 else 0\n            self.flop_counts[\'Global\'][packet] += int(flops)\n            return out\n        if name in {\'bernoulli\', \'bernoulli_\', \'uniform_\', \'normal_\'}:\n            return out\n        if name in {\'dropout\', \'native_dropout\'}:\n            probability = args[1] if len(args) > 1 else kwargs[\'p\']\n            training = args[2] if len(args) > 2 else kwargs.get(\'train\', True)\n            flops = 2 * args[0].numel() if training and 0 < probability < 1 else 0\n        elif name == \'native_dropout_backward\':\n            flops = 2 * args[0].numel()\n        elif name == \'gelu_backward\':\n            approximation = kwargs.get(\'approximate\', \'none\')\n            flops = args[0].numel() * (18 if approximation == \'tanh\' else 11)\n        elif name == \'native_layer_norm_backward\':\n            x, shape, weight, mask = (args[1], args[2], args[5], args[7])\n            elements = x.numel()\n            width = prod(shape)\n            rows = elements // width\n            flops = 2 * elements if mask[0] or mask[1] else 0\n            if mask[0]:\n                flops += (7 + int(weight is not None)) * elements\n            if mask[1]:\n                flops += elements + (elements - width)\n            if mask[2]:\n                flops += elements - width\n        elif name in {\'_log_softmax\', \'log_softmax\'}:\n            flops = 4 * args[0].numel()\n        elif name == \'_log_softmax_backward_data\':\n            grad, dim = (args[0], args[2])\n            flops = 4 * grad.numel() - grad.numel() // grad.shape[dim]\n        elif name == \'_softmax_backward_data\':\n            grad, dim = (args[0], args[2])\n            flops = 4 * grad.numel() - grad.numel() // grad.shape[dim]\n        elif name in {\'nll_loss_forward\', \'nll_loss_backward\'}:\n            offset = int(name == \'nll_loss_backward\')\n            weight = args[2 + offset] if len(args) > 2 + offset else kwargs.get(\'weight\')\n            reduction = args[3 + offset] if len(args) > 3 + offset else kwargs.get(\'reduction\', 1)\n            targets = args[1 + offset].numel()\n            if weight is not None:\n                self.uncovered_ops.add(str(packet) + \' (weighted NLL not covered)\')\n                return out\n            if offset:\n                flops = targets * (1 + int(reduction == 1))\n            else:\n                flops = targets\n                if reduction != 0:\n                    flops += max(targets - 1, 0) + int(reduction == 1)\n        elif name.startswith(\'_foreach_\'):\n            op = name[len(\'_foreach_\'):].rstrip(\'_\')\n            tensors = outputs or [x for x in tree_leaves(args[0]) if isinstance(x, torch.Tensor) and x.is_floating_point()]\n            elements = sum((x.numel() for x in tensors))\n            if op in {\'mul\', \'div\', \'sqrt\', \'rsqrt\', \'reciprocal\', \'neg\', \'pow\'}:\n                flops = elements\n            elif op in {\'add\', \'sub\'}:\n                flops = elements * (1 + int(kwargs.get(\'alpha\', 1) != 1))\n            elif op == \'lerp\':\n                flops = 3 * elements\n            elif op in {\'addcmul\', \'addcdiv\'}:\n                value = args[3] if len(args) > 3 else kwargs.get(\'value\', 1)\n                if isinstance(value, (list, tuple)):\n                    flops = sum((t.numel() * (2 + int(v != 1)) for t, v in zip(tensors, value)))\n                elif isinstance(value, torch.Tensor):\n                    self.uncovered_ops.add(str(packet) + \' (tensor scale not covered)\')\n                    return out\n                else:\n                    flops = elements * (2 + int(value != 1))\n            elif op in {\'copy\', \'zero\', \'maximum\', \'minimum\'}:\n                return out\n            else:\n                self.uncovered_ops.add(str(packet))\n                return out\n        elif name in {\'lerp\', \'lerp_\'}:\n            flops = 3 * n\n        elif name in {\'addcmul\', \'addcmul_\', \'addcdiv\', \'addcdiv_\'}:\n            flops = n * (2 + int(kwargs.get(\'value\', 1) != 1))\n        elif name in {\'sqrt_\', \'rsqrt_\', \'reciprocal_\', \'neg_\', \'pow_\'}:\n            flops = n\n        else:\n            if not outputs and name not in self.zero_flop_ops:\n                if any((isinstance(x, torch.Tensor) and x.is_floating_point() for x in tree_leaves(args))):\n                    self.uncovered_ops.add(str(packet))\n                return out\n            return super()._count_flops(packet, out, args, kwargs)\n        self.flop_counts[\'Global\'][packet] += int(flops)\n        return out\n\n\nclass RoutedFlopCounter(TrainingFlopCounter):\n    zero_flop_ops = TrainingFlopCounter.zero_flop_ops | {\'one_hot\', \'scatter\', \'scatter_\', \'index_put\', \'index_put_\', \'_index_put_impl_\', \'new_empty_strided\', \'empty_permuted\', \'clamp\', \'clamp_\', \'clamp_min\', \'clamp_min_\', \'clamp_max\', \'clamp_max_\', \'exponential\', \'exponential_\', \'bernoulli\', \'bernoulli_\'}\n\n    def _count_flops(self, packet, out, args, kwargs):\n        name = str(packet).split(\'.\')[-1]\n        outputs = [x for x in tree_leaves(out) if isinstance(x, torch.Tensor) and x.is_floating_point()]\n        if name in {\'index_put\', \'index_put_\', \'_index_put_impl_\'}:\n            accumulate = args[3] if len(args) > 3 else kwargs.get(\'accumulate\', False)\n            if accumulate:\n                self.flop_counts[\'Global\'][packet] += args[2].numel()\n            return out\n        if name in {\'sigmoid\', \'sigmoid_\'}:\n            flops = 4 * args[0].numel()\n        elif name == \'sigmoid_backward\':\n            flops = 3 * args[0].numel()\n        elif name == \'_thnn_fused_gru_cell\':\n            if len(args) < 5 or args[3] is None or args[4] is None:\n                self.uncovered_ops.add(str(packet) + \' (bias-free GRU not covered)\')\n                return out\n            flops = 23 * args[2].numel()\n        elif name == \'_thnn_fused_gru_cell_backward\':\n            grad = args[0]\n            has_bias = args[2] if len(args) > 2 else kwargs.get(\'has_bias\', False)\n            flops = 16 * grad.numel()\n            if has_bias:\n                flops += 6 * grad.shape[-1] * max(grad.shape[0] - 1, 0)\n        elif name in {\'std\', \'var\', \'std_mean\', \'var_mean\'}:\n            n = args[0].numel()\n            rows = outputs[0].numel()\n            flops = 4 * n + (rows if name.startswith(\'std\') else 0)\n        elif name == \'linalg_vector_norm\':\n            order = args[1] if len(args) > 1 else kwargs.get(\'ord\', 2)\n            if order != 2:\n                self.uncovered_ops.add(str(packet) + \' (non-L2 norm not covered)\')\n                return out\n            flops = 2 * args[0].numel()\n        elif name == \'_foreach_norm\':\n            order = args[1] if len(args) > 1 else kwargs.get(\'ord\', 2)\n            if order != 2:\n                self.uncovered_ops.add(str(packet) + \' (non-L2 norm not covered)\')\n                return out\n            flops = 2 * sum((t.numel() for t in args[0]))\n        elif name in {\'log_\', \'exp_\'}:\n            flops = args[0].numel()\n        elif name in {\'one_hot\', \'scatter\', \'scatter_\', \'clamp\', \'clamp_\', \'clamp_min\', \'clamp_min_\', \'clamp_max\', \'clamp_max_\', \'exponential\', \'exponential_\'}:\n            return out\n        else:\n            return super()._count_flops(packet, out, args, kwargs)\n        self.flop_counts[\'Global\'][packet] += int(flops)\n        return out\n'


In [ ]:
# Inference-only baselines and persistent, protocol-aware quality/cost archive.
ANALYSIS_VERSION = 2
RUN_NEW_BASELINE_MEASUREMENTS = True
EARLY_EXIT_CALIBRATION_STEPS = 5
EARLY_EXIT_LAYERS = (2, 4, 6, 8, 10, 12)
RANDOM_ROUTING_SEEDS = (101, 202, 303)
BUDGET_MATCH_TOLERANCE_PP = 3.0  # Percentage points of the full expert's FLOPs.
ARCHIVE_PATH = BASELINES_PATH / 'ExperimentalAnalysis' / 'quality_cost_archive.json'


def qc_hash(value):
    import hashlib
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'),
                                     ensure_ascii=False, allow_nan=False).encode('utf-8')).hexdigest()


def qc_file_hash(path):
    import hashlib
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def qc_save_archive(archive):
    import os
    ARCHIVE_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary = ARCHIVE_PATH.with_suffix('.json.partial')
    with temporary.open('w', encoding='utf-8') as handle:
        json.dump(archive, handle, ensure_ascii=False, allow_nan=False)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, ARCHIVE_PATH)
    if os.name == 'posix':
        fd = os.open(str(ARCHIVE_PATH.parent), os.O_RDONLY | os.O_DIRECTORY)
        try:
            os.fsync(fd)
        finally:
            os.close(fd)


def qc_load_archive():
    if not ARCHIVE_PATH.exists():
        return {'schema_version': 1, 'measurements': {}, 'points': {}}
    result = read_json(ARCHIVE_PATH)
    if result.get('schema_version') != 1:
        raise ValueError('Versione archivio qualità/costo non supportata')
    return result


def qc_configuration(item):
    parameters = item.get('experiment', {}).get('parameters', {})
    ignored = {'SEED', 'SEED_INDEX', 'RUN_SEEDS', 'RUN_STARTED_AT', 'RUN_TIMESTAMP',
               'EXPERIMENT_TIMESTAMP', 'RUN_STATE_NAMES', 'DEVICE', 'PIN_MEMORY', 'NUM_WORKERS',
               'SHOW_BATCH_PROGRESS', 'ROUTED_BASELINES_DIRECTORY', 'LATENCY_REPETITIONS',
               'LATENCY_WARMUP_BATCHES', 'BOOTSTRAP_SEED', 'BOOTSTRAP_RESAMPLES'}
    return {key: value for key, value in parameters.items()
            if key not in ignored and not key.endswith('_PATH')}


def qc_trace(folder, item, task, split):
    import hashlib
    record = item['splits'][f'{task}_{split}']
    info = record['per_input']
    path = folder / 'routing_per_input' / Path(info['jsonl_path']).name
    fields = ('example_id', 'input_sha256', 'input_ids', 'attention_mask', 'token_type_ids',
              'label', 'prediction')
    rows, digest, ids = [], hashlib.sha256(), set()
    with path.open('rb') as handle:
        for line in handle:
            digest.update(line)
            original = json.loads(line)
            row = {key: original[key] for key in fields}
            if original['task'] != task or original['split'] != split or row['example_id'] in ids:
                raise ValueError(f'Tracce non allineate o duplicate: {path}')
            ids.add(row['example_id'])
            active_tokens = [(int(i), int(t)) for i, t, m in zip(
                row['input_ids'], row['token_type_ids'], row['attention_mask']) if m]
            token_hash = hashlib.sha256(json.dumps(active_tokens, separators=(',', ':')).encode()).hexdigest()
            if token_hash != row['input_sha256']:
                raise ValueError(f'Identità token non valida: {path}')
            rows.append(row)
    if info.get('sha256') and digest.hexdigest() != info['sha256']:
        raise ValueError(f'Tracce modificate rispetto al manifest: {path}')
    if len(rows) != record['metrics']['num_samples']:
        raise ValueError(f'Tracce incomplete: {path}')
    data_id = qc_hash([(r['example_id'], r['input_sha256'], r['label']) for r in rows])
    cost = record['inference_cost']
    convention = {k: v for k, v in cost['convention'].items() if k != 'router'}
    protocol_id = qc_hash({'data': data_id, 'input': cost['input'], 'convention': convention,
                           'method': cost['method'], 'torch': cost.get('torch_version'),
                           'transformers': cost.get('transformers_version'),
                           'expert': item['fixed_expert_test'][task]['checkpoint_sha256']})
    return {'rows': rows, 'data_id': data_id, 'protocol_id': protocol_id, 'task': task,
            'split': split, 'cost_template': cost, 'path': str(path)}


def qc_metrics(labels, predictions, task):
    classes = 2 if task == 'sst' else 3
    matrix = [[0] * classes for _ in range(classes)]
    for label, predicted in zip(labels, predictions):
        matrix[int(label)][int(predicted)] += 1
    raw = metrics_from_result({'confusion_matrix': matrix, 'num_samples': len(labels)}, task)
    return {'accuracy': raw['accuracy'], 'f1': raw['f1'], 'num_samples': raw['n'],
            'f1_average': 'binary' if task == 'sst' else 'macro'}, matrix


def qc_point(archive, context, result, method, policy, source, seed=None, random_seed=None):
    # Identity is independent of output folder location and never includes metric values.
    identity = {'version': ANALYSIS_VERSION, 'protocol': context['protocol_id'], 'method': method,
                'policy': policy, 'source': source, 'seed': seed, 'random_seed': random_seed}
    key = qc_hash(identity)
    archive['points'][key] = {
        'id': key, 'task': context['task'], 'split': 'test', 'method': method,
        'protocol_id': context['protocol_id'], 'data_id': context['data_id'],
        'configuration': policy, 'configuration_id': qc_hash({'method': method, 'policy': policy}),
        'source': source, 'seed': seed, 'random_seed': random_seed,
        'trace_path': context['path'],
        'metrics': result['metrics'], 'flops_per_example': result['inference_cost']['flops_per_example'],
        'coverage': result['inference_cost']['coverage']['status'],
    }
    return key




def qc_pareto(points):
    return [p for p in points if not any(
        q['cost'] <= p['cost'] and q['f1'] >= p['f1']
        and (q['cost'] < p['cost'] or q['f1'] > p['f1']) for q in points)]


def qc_group_points(points):
    import statistics
    groups = {}
    for point in points:
        if point['coverage'] != 'complete_for_convention' or point['flops_per_example'] is None:
            continue
        groups.setdefault((point['method'], point['configuration_id']), []).append(point)
    result = []
    for (method, config), observations in groups.items():
        # Average random-policy repeats within each trained checkpoint first.
        repetitions = {}
        for point in observations:
            repetitions.setdefault((point['source'], point['seed']), []).append(point)
        values = [(statistics.mean(p['metrics']['f1'] for p in group),
                   statistics.mean(p['flops_per_example'] for p in group)) for group in repetitions.values()]
        f1s, costs = zip(*values)
        spread_f1s, spread_costs = f1s, costs
        spread_kind = 'seed di addestramento'
        if len(values) == 1 and len(observations) > 1:
            spread_f1s = [p['metrics']['f1'] for p in observations]
            spread_costs = [p['flops_per_example'] for p in observations]
            spread_kind = 'semi del routing casuale'
        result.append({'method': method, 'config': config, 'f1': statistics.mean(f1s),
                       'cost': statistics.mean(costs), 'f1_std': statistics.stdev(spread_f1s) if len(spread_f1s) > 1 else None,
                       'cost_std': statistics.stdev(spread_costs) if len(spread_costs) > 1 else None,
                       'spread_kind': spread_kind,
                       'n_models': len(values), 'n_observations': len(observations),
                       'settings': observations[0]['configuration']})
    return result




def qc_tensors(context, engine):
    torch = engine['torch']
    rows = context['rows']
    return tuple(torch.tensor([r[key] for r in rows], dtype=torch.long)
                 for key in ('input_ids', 'attention_mask', 'token_type_ids'))


def qc_measure(archive, engine, context, source, policy, forward, tensors, random_seed=None):
    import time
    torch = engine['torch']
    key = qc_hash({'version': ANALYSIS_VERSION, 'protocol': context['protocol_id'],
                   'source': source, 'policy': policy, 'random_seed': random_seed})
    if key in archive['measurements']:
        return archive['measurements'][key]
    size = context['cost_template']['input']['batch_size']
    n = len(context['rows'])
    if tensors[0].shape[0] != n:
        raise ValueError('Numero di input incoerente')
    predictions, total, uncovered, operators = [], 0, set(), {}
    started = time.monotonic()
    print(f"Misura {context['task']}/{context['split']}: {policy} seed={random_seed}", flush=True)
    with torch.inference_mode():
        for first in range(0, n, size):
            batch = tuple(x[first:first + size].to(engine['device']) for x in tensors)
            with engine['RoutedFlopCounter']() as counter:
                logits = forward(batch, first)
            predictions.extend(logits.argmax(-1).cpu().tolist())
            total += int(counter.get_total_flops())
            uncovered.update(counter.uncovered_ops)
            for op, value in counter.get_flop_counts().get('Global', {}).items():
                operators[str(op)] = operators.get(str(op), 0) + int(value)
            if first and (first // size) % 25 == 0:
                print(f'  {first}/{n} esempi', flush=True)
    if uncovered or total <= 0:
        raise RuntimeError(f'Conteggio FLOPs incompleto per {policy}: {sorted(uncovered)}')
    labels = [r['label'] for r in context['rows']]
    metrics, matrix = qc_metrics(labels, predictions, context['task'])
    import copy
    cost = copy.deepcopy(context['cost_template'])
    cost.update(total_flops=total, flops_per_example=total / n, num_examples=n,
                scope='executed baseline forward; confidence classifiers included; no training or data transfers')
    cost['convention']['router'] = 'no learned router; random indices precomputed; early-exit heads and softmax counted when executed'
    cost['coverage'] = {'status': 'complete_for_convention', 'uncovered_operators': [],
                        'counted_flops': total, 'counted_flops_by_operator': operators}
    result = {'metrics': metrics, 'confusion_matrix': matrix, 'inference_cost': cost,
              'predictions': predictions, 'policy': policy, 'random_seed': random_seed,
              'elapsed_seconds': time.monotonic() - started}
    archive['measurements'][key] = result
    qc_save_archive(archive)
    return result


def qc_expert_forward(engine, backbone, head, batch, keep=None, threshold=None):
    torch = engine['torch']
    ids, mask, types = batch
    hidden = backbone.embeddings(input_ids=ids, token_type_ids=types)
    extended = backbone.get_extended_attention_mask(mask, ids.shape, dtype=hidden.dtype)
    if threshold is None:
        for index, layer in enumerate(backbone.encoder.layer):
            if keep is None or index in keep:
                hidden = engine['extract_layer_hidden_states'](layer(hidden, attention_mask=extended))
        return head(backbone.pooler(hidden) if backbone.pooler is not None else hidden[:, 0, :])
    # Shared existing final head; per-example exits physically remove examples from later layers.
    active = torch.arange(ids.shape[0], device=ids.device)
    output = None
    depth = len(backbone.encoder.layer)
    for index, layer in enumerate(backbone.encoder.layer):
        hidden = engine['extract_layer_hidden_states'](layer(hidden, attention_mask=extended))
        if index + 1 not in EARLY_EXIT_LAYERS and index + 1 != depth:
            continue
        logits = head(backbone.pooler(hidden) if backbone.pooler is not None else hidden[:, 0, :])
        if output is None:
            output = logits.new_empty((ids.shape[0], logits.shape[1]))
        leave = (logits.softmax(-1).amax(-1) >= threshold) if index + 1 < depth else torch.ones_like(active, dtype=torch.bool)
        output[active[leave]] = logits[leave]
        if bool(leave.all()):
            return output
        active, hidden, extended = active[~leave], hidden[~leave], extended[~leave]
    return output










def qc_resolve_model(path, folder=None):
    path = Path(path)
    if path.is_file() or path.is_dir():
        return path
    if folder is not None:
        candidate = BASELINES_PATH / 'RoutedModel' / 'Models' / folder.parent.name / folder.name / path.name
    else:
        candidate = BASELINES_PATH / 'TrainingOfUncasedModel' / 'Models' / path.name
    if candidate.exists():
        return candidate
    raise FileNotFoundError(f'Checkpoint mancante: {path}')


def qc_average_results(results):
    import statistics
    if len(results) == 1:
        return results[0]
    return {'metrics': {k: statistics.mean(r['metrics'][k] for r in results) for k in ('f1', 'accuracy')},
            'inference_cost': {'flops_per_example': statistics.mean(r['inference_cost']['flops_per_example'] for r in results)},
            'f1_std_random': statistics.stdev(r['metrics']['f1'] for r in results), 'random_repeats': len(results)}




def qc_engine():
    namespace = {'__name__': 'analysis_inference_components'}
    exec(INFERENCE_COMPONENT_SOURCE, namespace)
    import transformers
    namespace['transformers_version'] = str(transformers.__version__)
    namespace['device'] = 'cuda'
    return namespace


def qc_expert_random_choices(n, depth, skip_probability, seed):
    import numpy as np
    rng = np.random.default_rng(seed)
    choices = rng.integers(0, 2, size=(n, depth))
    choices[rng.random((n, depth)) < skip_probability] = 2
    return choices


def qc_expert_random_forward(engine, experts, head, task, batch, choices):
    torch = engine['torch']
    ids, mask, types = batch
    own = experts[task]
    hidden = own.embeddings(input_ids=ids, token_type_ids=types)
    extended = own.get_extended_attention_mask(mask, ids.shape, dtype=hidden.dtype)
    for depth in range(len(own.encoder.layer)):
        updated = hidden.clone()
        for branch, name in enumerate(('sst', 'mnli')):
            selected = choices[:, depth] == branch
            if bool(selected.any()):
                updated[selected] = engine['extract_layer_hidden_states'](
                    experts[name].encoder.layer[depth](hidden[selected], attention_mask=extended[selected]))
        hidden = updated
    return head(own.pooler(hidden) if own.pooler is not None else hidden[:, 0, :])


def qc_sample_cost(engine, forward, tensors):
    torch = engine['torch']
    batch = tuple(x[:min(32, len(x))].to(engine['device']) for x in tensors)
    with torch.inference_mode(), engine['RoutedFlopCounter']() as counter:
        forward(batch, 0)
    if counter.uncovered_ops:
        raise RuntimeError(f'Calibrazione FLOPs incompleta: {counter.uncovered_ops}')
    return counter.get_total_flops() / len(batch[0])


def qc_run(archive):
    import statistics
    contexts, comparisons, current_protocols = {}, [], {}
    for folder, item in GROUP_RUNS:
        for task in TASKS:
            for split in ('validation', 'test'):
                contexts[(folder.name, task, split)] = qc_trace(folder, item, task, split)
            test = contexts[(folder.name, task, 'test')]
            current_protocols[task] = test['protocol_id']
            config = {'parameters': qc_configuration(item), 'experts': {
                t: item['fixed_expert_test'][t]['checkpoint_sha256'] for t in TASKS}}
            qc_point(archive, test, item['splits'][f'{task}_test'], 'Routed', config,
                     item['checkpoint']['sha256'], item['experiment_group']['seed'])
            expert = item['fixed_expert_test'][task]
            qc_point(archive, test, expert, 'Esperto completo', {'expert': expert['checkpoint_sha256']},
                     qc_hash(expert['checkpoint_sha256']))
    qc_save_archive(archive)
    report_key = qc_hash({'version': ANALYSIS_VERSION,
                          'checkpoints': [item['checkpoint']['sha256'] for _, item in GROUP_RUNS],
                          'protocols': current_protocols, 'early_steps': EARLY_EXIT_CALIBRATION_STEPS,
                          'early_layers': EARLY_EXIT_LAYERS, 'random_seeds': RANDOM_ROUTING_SEEDS})
    saved = archive.get('comparison_sets', {}).get(report_key)
    if saved is not None:
        print('Confronti già misurati: riutilizzo archivio, nessuna inferenza necessaria.', flush=True)
        return saved, current_protocols, ['Confronti completi recuperati dall’archivio. Nessuna nuova inferenza.']
    if not RUN_NEW_BASELINE_MEASUREMENTS:
        return comparisons, current_protocols, ['Nuove misure disattivate: grafici costruiti dai punti già archiviati.']
    engine = qc_engine()
    torch = engine['torch']
    if engine['device'] == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('Le nuove misure richiedono la GPU. Usare RUN_NEW_BASELINE_MEASUREMENTS=False per il solo report.')
    from transformers import AutoModel
    first_folder, first_item = GROUP_RUNS[0]
    recorded = first_item['splits']['sst_test']['inference_cost']
    if (recorded.get('torch_version') != str(torch.__version__)
            or recorded.get('transformers_version') != engine['transformers_version']):
        raise RuntimeError('Versioni PyTorch/Transformers diverse dalle misure routed: usare lo stesso ambiente per il confronto.')
    if recorded['input'].get('dtype') != 'torch.float32':
        raise RuntimeError('Questa valutazione richiede esperti e misure routed in float32.')
    experts, heads, identities = {}, {}, {}
    for task in TASKS:
        path = qc_resolve_model(first_item['expert_directories'][task])
        identities[task] = engine['expert_weight_identity'](path)
        if identities[task] != first_item['fixed_expert_test'][task]['checkpoint_sha256']:
            raise ValueError('Gli esperti su disco non coincidono con quelli usati dal routed')
        experts[task] = AutoModel.from_pretrained(path / 'bert_backbone', local_files_only=True).to(engine['device']).eval()
        parameters = first_item['experiment']['parameters']
        heads[task] = engine['ClassifierHead'](input_dim=parameters['HIDDEN_SIZE'], hidden_dim=parameters['HEAD_HIDDEN_SIZE'],
                                              num_classes=2 if task == 'sst' else 3,
                                              dropout_prob=parameters['DROPOUT_PROBABILITY']).to(engine['device']).eval()
        heads[task].load_state_dict(torch.load(path / 'classifier.pt', map_location='cpu', weights_only=True))
    if len(experts['sst'].encoder.layer) != len(experts['mnli'].encoder.layer):
        raise ValueError('Il routing casuale richiede esperti con lo stesso numero di layer')
    for task in TASKS:
        val, test = (contexts[(first_folder.name, task, split)] for split in ('validation', 'test'))
        for folder, _ in GROUP_RUNS:
            if any(contexts[(folder.name, task, split)]['protocol_id'] != contexts[(first_folder.name, task, split)]['protocol_id']
                   for split in ('validation', 'test')):
                raise ValueError('I seed del gruppo non usano lo stesso protocollo/dataset')
        tensors = {'validation': qc_tensors(val, engine), 'test': qc_tensors(test, engine)}
        # One budget per task: the current configuration's mean validation cost across seeds.
        target = statistics.mean(item['splits'][f'{task}_validation']['inference_cost']['flops_per_example'] for _,item in GROUP_RUNS)
        routed = qc_average_results([item['splits'][f'{task}_test'] for _,item in GROUP_RUNS])
        routed['f1_std_seeds'] = statistics.stdev(item['splits'][f'{task}_test']['metrics']['f1'] for _,item in GROUP_RUNS) if len(GROUP_RUNS)>1 else None
        own, head = experts[task], heads[task]
        depth = len(own.encoder.layer)
        source = qc_hash(identities[task])
        full = qc_sample_cost(engine, lambda batch, first: qc_expert_forward(engine, own, head, batch), tensors['validation'])
        fixed = qc_sample_cost(engine, lambda batch, first: qc_expert_forward(engine, own, head, batch, keep=[]), tensors['validation'])
        count = max(1., min(float(depth), depth*(target-fixed)/(full-fixed)))
        # Whole static layers cannot always match a fractional budget: measure the two neighbours.
        counts = sorted({max(1, math.floor(count)), min(depth, math.ceil(count))})
        static_candidates = []
        for keep_count in counts:
            keep = [0] if keep_count == 1 else [round(i*(depth-1)/(keep_count-1)) for i in range(keep_count)]
            policy = {'method':'static', 'keep':keep, 'selection':'evenly spaced; two layer counts bracketing validation budget'}
            forward = lambda batch, first, keep=keep: qc_expert_forward(engine, own, head, batch, keep=keep)
            candidate = {'policy':policy}
            for context in (val,test):
                candidate[context['split']] = qc_measure(archive,engine,context,source,policy,forward,tensors[context['split']])
            qc_point(archive,test,candidate['test'],'Layer dropping statico',{'policy':policy,'expert':source},source)
            static_candidates.append(candidate)
        # Reproduce the original expert once on the exact saved test inputs.
        reference_policy = {'method':'full_expert_check'}
        result = qc_measure(archive,engine,test,source,reference_policy,
                            lambda batch,first: qc_expert_forward(engine,own,head,batch),tensors['test'])
        import csv
        with (GROUP_DIR/f'expert_{task}_predictions.csv').open(encoding='utf-8') as handle:
            expected = {r['example_id']:int(r['prediction']) for r in csv.DictReader(handle)}
        if result['predictions'] != [expected[r['example_id']] for r in test['rows']]:
            raise RuntimeError('Il forward completo non riproduce le predizioni dell’esperto salvato')
        # Random routing: SST / MNLI / skip, with the task expert's existing embedding, pooler and head.
        random_source = qc_hash(identities)
        def random_endpoint(skip):
            n = min(32,len(tensors['validation'][0]))
            routes = torch.as_tensor(qc_expert_random_choices(n,depth,skip,0),device=engine['device'])
            return qc_sample_cost(engine,lambda batch,first: qc_expert_random_forward(engine,experts,head,task,batch,routes),tensors['validation'])
        maximum, minimum = random_endpoint(0.), random_endpoint(1.)
        if maximum <= minimum:
            raise ValueError('Intervallo FLOPs del routing casuale non valido')
        probability = min(1.,max(0.,(maximum-target)/(maximum-minimum)))
        policy = {'method':'random_expert_routing','skip_probability':probability,
                  'active_branch_probability':{'sst':.5,'mnli':.5}, 'embedding_pooler_head':task}
        random_results = {'validation':[], 'test':[]}
        for seed in RANDOM_ROUTING_SEEDS:
            for context in (val,test):
                routes = torch.as_tensor(qc_expert_random_choices(len(context['rows']),depth,probability,seed),device=engine['device'])
                forward = lambda batch,first: qc_expert_random_forward(engine,experts,head,task,batch,routes[first:first+len(batch[0])])
                measured = qc_measure(archive,engine,context,random_source,policy,forward,tensors[context['split']],seed)
                random_results[context['split']].append(measured)
                if context['split']=='test':
                    qc_point(archive,test,measured,'Routing casuale',{'policy':policy,'experts':identities},random_source,random_seed=seed)
                del routes
        random_candidate = {'policy':policy, **{split:qc_average_results(values) for split,values in random_results.items()}}
        # A short bounded bisection on validation cost, not a hyperparameter search on test quality.
        early_candidates=[]
        def evaluate_threshold(threshold):
            policy={'method':'early_exit_shared_head','threshold':threshold,'layers':list(EARLY_EXIT_LAYERS)}
            forward=lambda batch,first: qc_expert_forward(engine,own,head,batch,threshold=threshold)
            candidate={'policy':policy,'validation':qc_measure(archive,engine,val,source,policy,forward,tensors['validation'])}
            early_candidates.append(candidate)
            return candidate['validation']['inference_cost']['flops_per_example']
        low,high=0.,1.01
        evaluate_threshold(low);evaluate_threshold(high)
        for _ in range(EARLY_EXIT_CALIBRATION_STEPS):
            middle=(low+high)/2
            measured_cost=evaluate_threshold(middle)
            if measured_cost<target:low=middle
            else:high=middle
        early=min(early_candidates,key=lambda c:(abs(c['validation']['inference_cost']['flops_per_example']-target),
                                                -c['validation']['metrics']['f1']))
        threshold=early['policy']['threshold']
        early['test']=qc_measure(archive,engine,test,source,early['policy'],
                                 lambda batch,first: qc_expert_forward(engine,own,head,batch,threshold=threshold),tensors['test'])
        qc_point(archive,test,early['test'],'Early exit — testa condivisa',{'policy':early['policy'],'expert':source},source)
        row={'task':task,'seed':'media dei seed ' + ', '.join(str(item['experiment_group']['seed']) for _,item in GROUP_RUNS),
             'routed':routed,'reference_test_flops':first_item['fixed_expert_test'][task]['inference_cost']['flops_per_example'],
             'validation_target':target,'baselines':{}}
        for candidate in static_candidates:
            row['baselines'][f"Statico — {len(candidate['policy']['keep'])}/{depth} layer"] = candidate
        row['baselines']['Routing casuale']=random_candidate
        row['baselines']['Early exit — testa condivisa']=early
        comparisons.append(row)
        qc_save_archive(archive)
    archive.setdefault('comparison_sets',{})[report_key]=comparisons
    qc_save_archive(archive)
    del experts,heads
    import gc
    gc.collect();torch.cuda.empty_cache()
    return comparisons,current_protocols,[]


def qc_svg_plot(groups, task):
    import io
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 5.5))
    palette = {'Routed': '#2563eb', 'Esperto completo': '#111827', 'Layer dropping statico': '#059669',
               'Routing casuale': '#d97706', 'Early exit — testa condivisa': '#9333ea'}
    for method in dict.fromkeys(p['method'] for p in groups):
        rows = [p for p in groups if p['method'] == method]
        ax.errorbar([p['cost'] / 1e9 for p in rows], [100*p['f1'] for p in rows],
                    xerr=[(p['cost_std'] or 0) / 1e9 for p in rows], yerr=[100*(p['f1_std'] or 0) for p in rows],
                    fmt='o', capsize=3, label=method, color=palette.get(method), alpha=.85)
    front = sorted(qc_pareto(groups), key=lambda p: p['cost'])
    if front:
        ax.scatter([p['cost']/1e9 for p in front], [100*p['f1'] for p in front],
                   facecolors='none', edgecolors='#dc2626', s=150, linewidths=1.5, label='Non dominati tra i punti misurati')
    ax.set(xlabel='GFLOPs per esempio (costo effettivamente misurato)', ylabel='F1 (%)', title=f'{task.upper()} — qualità e costo sul test')
    ax.grid(alpha=.2); ax.legend(fontsize=8); fig.tight_layout()
    stream = io.StringIO(); fig.savefig(stream, format='svg'); plt.close(fig)
    svg = stream.getvalue()
    return svg[svg.index('<svg'):]


def append_quality_cost_report(parts, archive, comparisons, protocols, messages):
    parts.append('<h2>Confronti con baseline a budget comparabile</h2>')
    parts.append('<p>Tutte le nuove misure usano checkpoint già addestrati, senza aggiornare pesi. '
                 'Il budget è il costo medio del routed sulla validation tra i seed della configurazione corrente, separatamente per task. '
                 'Le politiche vengono regolate per avvicinarsi a quel costo; la qualità viene poi misurata sugli stessi input di test del routed. '
                 'Non vengono esplorati altri budget. '
                 'Questi sono confronti post-training: non rappresentano metodi riaddestrati o distillati.</p>')
    parts.append('<ul><li><strong>Statico:</strong> layer dell’esperto distribuiti uniformemente in profondità, con la stessa maschera per ogni input. '
                 'Si misurano i due numeri interi di layer più vicini al budget: uno inferiore e uno superiore, quando distinti.</li>'
                 '<li><strong>Casuale:</strong> a ogni profondità si salta un layer oppure si sceglie con pari probabilità il layer dell’esperto SST o MNLI. '
                 'La probabilità di salto deriva dal budget di validation. Embedding, pooler e testa sono quelli dell’esperto del task; '
                 'non si usano router appreso o layer merged. Si ripete con tre semi casuali. '
                 'Questa baseline confronta il routed con una combinazione casuale degli esperti originali; non isola il solo effetto del router a pesi identici.</li>'
                 '<li><strong>Early exit:</strong> pooler e testa finale dell’esperto riutilizzati alle profondità intermedie. '
                 'Uscita per esempio sulla confidenza softmax, senza testine addestrate appositamente. '
                 'Il costo include tutte le teste e le softmax effettivamente eseguite.</li></ul>')
    for message in messages:
        parts.append('<p><strong>Stato:</strong> ' + h(message) + '</p>')
    for row in comparisons:
        r = row['routed']; rc = r['inference_cost']['flops_per_example']
        rf1 = fmt(100*r['metrics']['f1'], 3)
        if r.get('f1_std_seeds') is not None:
            rf1 += ' ± ' + fmt(100*r['f1_std_seeds'], 3) + ' (seed addestramento)'
        rows = [['Routed', rf1, '0', fmt(rc/1e9, 3), '0', 'Riferimento', '—']]
        for name, candidate in row['baselines'].items():
            if candidate is None:
                rows.append([name, 'N/D', 'N/D', 'N/D', 'N/D', 'Misura assente', '—'])
                continue
            result = candidate['test']; bc = result['inference_cost']['flops_per_example']
            gap = 100*(bc - rc)/row['reference_test_flops']
            matched = abs(gap) <= BUDGET_MATCH_TOLERANCE_PP
            note = 'Costi vicini' if matched else 'Costi non coincidenti: non è un confronto a parità stretta'
            f1_text = fmt(100*result['metrics']['f1'], 3)
            if 'f1_std_random' in result:
                f1_text += ' ± ' + fmt(100*result['f1_std_random'], 3) + ' (semi casuali)'
            rows.append([name, f1_text, fmt(100*(result['metrics']['f1']-r['metrics']['f1']), 3), fmt(bc/1e9, 3), fmt(gap, 2), note,
                         json.dumps(candidate['policy'], ensure_ascii=False)])
        parts.append(f'<h3>{row["task"].upper()} — {h(row["seed"])}</h3>')
        parts.append(f'<p>Budget di validation: {fmt(row["validation_target"]/1e9, 3)} GFLOPs per esempio.</p>')
        parts.append(table(['Metodo', 'F1 test %', 'Δ F1 rispetto al routed (punti %)', 'GFLOPs/esempio', 'Scarto costo / esperto (punti %)',
                            'Confrontabilità del budget', 'Configurazione scelta su validation'], rows))
    parts.append(f'<p>La vicinanza di costo è segnalata entro {BUDGET_MATCH_TOLERANCE_PP:g} punti percentuali '
                 'del costo dell’esperto. Layer interi e soglie discrete possono lasciare budget inutilizzato; '
                 'la tabella riporta il costo reale, senza interpolare risultati. '
                 'Un vantaggio a costi diversi non dimostra da solo la superiorità del routing.</p>')
    parts.append('<h2>Frontiera qualità/costo e archivio delle configurazioni</h2>')
    parts.append('<p>L’archivio conserva i punti delle esecuzioni analizzate e aggiorna quelli già presenti senza duplicarli. '
                 'I seed della stessa configurazione routed sono aggregati: non sono configurazioni diverse. '
                 'Le barre indicano la deviazione standard tra seed di addestramento per il routed e tra semi casuali per la baseline casuale. '
                 'Una sola configurazione routed produce un solo punto per task: altre configurazioni compariranno dopo le analisi future. '
                 'I cerchi rossi identificano la frontiera empirica sui valori medi, senza un test statistico di dominanza. '
                 'Sono mostrati soltanto punti con gli stessi esempi, protocollo FLOPs, input e riferimento esperto della run corrente.</p>')
    for task in TASKS:
        observations = [p for p in archive['points'].values() if p['task'] == task
                        and p['protocol_id'] == protocols.get(task)]
        excluded = sum(p['task'] == task and p['protocol_id'] != protocols.get(task) for p in archive['points'].values())
        groups = qc_group_points(observations)
        if not groups:
            parts.append(f'<p>{task.upper()}: nessun punto confrontabile disponibile.</p>')
            continue
        parts.append(qc_svg_plot(groups, task))
        front_ids = {p['config'] for p in qc_pareto(groups)}
        rows = [[p['method'], p['config'][:10], fmt(100*p['f1'], 3), fmt(p['cost']/1e9, 3),
                 p['n_models'], p['n_observations'], 'Sì' if p['config'] in front_ids else 'No',
                 json.dumps(p['settings'], ensure_ascii=False)] for p in sorted(groups, key=lambda p: p['cost'])]
        parts.append('<details><summary>Configurazioni e provenienza — ' + task.upper() + '</summary>'
                     + table(['Metodo', 'Configurazione', 'F1 media %', 'GFLOPs medi', 'Checkpoint', 'Misure',
                              'Non dominata', 'Parametri'], rows) + '</details>')
        parts.append(f'<p>{excluded} punti archiviati esclusi da questo grafico perché usano un protocollo diverso.</p>')
    parts.append('<p>Archivio persistente: <code>' + h(str(ARCHIVE_PATH)) + '</code>. '
                 'Le misure vengono salvate dopo ogni valutazione completa. Un rilancio riusa le misure compatibili; '
                 'l’eventuale valutazione interrotta viene rifatta.</p>')


In [ ]:

QC_ARCHIVE = qc_load_archive()
QC_COMPARISONS, QC_PROTOCOLS, QC_MESSAGES = [], {}, []
QC_FAILURE = None
try:
    QC_COMPARISONS, QC_PROTOCOLS, QC_MESSAGES = qc_run(QC_ARCHIVE)
except Exception as error:
    # Preserve the original report and explicitly mark incomplete experimental sections.
    QC_FAILURE = error
    QC_MESSAGES.append(f"MISURE NON COMPLETATE: {type(error).__name__}: {error}. "
                       "Le valutazioni già salvate saranno riutilizzate al prossimo avvio.")
    print(QC_MESSAGES[-1], flush=True)
    for task in TASKS:
        try:
            QC_PROTOCOLS[task] = qc_trace(GROUP_RUNS[0][0], GROUP_RUNS[0][1], task, 'test')['protocol_id']
        except Exception:
            pass


In [ ]:
# Presentation only. All model evaluations and stored measurements are unchanged.
import statistics


def h(value):
    return escape(str(value), quote=True)


def fmt(value, digits=3, signed=False):
    if value is None:
        return 'N/D'
    return format(value, ('+' if signed else '') + f'.{digits}f').replace('.', ',')


def table(headers, rows):
    head = ''.join(f'<th scope="col">{h(x)}</th>' for x in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{h(x)}</td>' for x in row) + '</tr>' for row in rows)
    return f'<div class="table-wrap"><table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'


def report_metrics(result, task):
    values = metrics_from_result(result, task)
    matrix = result.get('confusion_matrix', result.get('metrics', {}).get('confusion_matrix'))
    classes = [1] if task == 'sst' else list(range(3))
    precision, recall = [], []
    for index in classes:
        predicted = sum(row[index] for row in matrix)
        actual = sum(matrix[index])
        precision.append(matrix[index][index] / predicted if predicted else 0.)
        recall.append(matrix[index][index] / actual if actual else 0.)
    return {**values, 'precision': statistics.mean(precision), 'recall': statistics.mean(recall)}


def report_mean_std(values, scale=1):
    values = [v for v in values if v is not None]
    if not values:
        return 'N/D'
    mean = fmt(statistics.mean(values)*scale)
    return mean + (' ± ' + fmt(statistics.stdev(values)*scale) if len(values)>1 else ' (una misura)')


def report_cost(result):
    return measured_flops(result.get('inference_cost', {}))[0]


def report_saving(reference, cost):
    return 100*(1-cost/reference) if reference and cost is not None else None


def report_quality_sentence(delta):
    if abs(delta) < 0.0005:
        return 'F1 uguale al confronto alla precisione mostrata.'
    return f'Il routed ha {fmt(abs(delta))} punti F1 in ' + ('più.' if delta>0 else 'meno.')


def report_metric_note(task):
    if task == 'sst':
        return 'Precision, recall e F1 si riferiscono alla classe positiva (etichetta 1); accuracy su tutti gli esempi.'
    return 'Precision, recall e F1 sono macro: media delle tre classi; accuracy su tutti gli esempi.'


def report_routing_chart(record, task, seed):
    routing = record['routing']
    layers = routing['per_layer']
    colors = {'SST':'#2563eb','MNLI':'#d97706','MERGED':'#8b5cf6','IDENTITY':'#64748b'}
    labels = {'SST':'Esperto SST','MNLI':'Esperto MNLI','MERGED':'Merged','IDENTITY':'Salto'}
    left, top, width, height = 50, 28, 570, 200
    step = width/len(layers)
    parts = [f'<svg class="routing-chart" viewBox="0 0 660 310" role="img" aria-label="Routing {task.upper()}, seed {seed}, validation">',
             f'<title>{task.upper()} — seed {seed} — scelte per layer sulla validation</title>']
    for percent in (0,25,50,75,100):
        y = top+height*(1-percent/100)
        parts.append(f'<line x1="{left}" x2="{left+width}" y1="{y}" y2="{y}" stroke="#dbe3ed"/>'
                     f'<text x="{left-6}" y="{y+4}" text-anchor="end" font-size="11">{percent}%</text>')
    for index, layer in enumerate(layers):
        bottom = top+height
        for choice in CHOICES:
            percent = layer['hard_choice_percentages'][choice]
            bar_height = height*percent/100
            bottom -= bar_height
            parts.append(f'<rect x="{left+index*step+5}" y="{bottom}" width="{step-10}" height="{bar_height}" fill="{colors[choice]}">'
                         f'<title>Layer {layer["layer"]}: {labels[choice]} {fmt(percent,2)}%</title></rect>')
        parts.append(f'<text x="{left+(index+.5)*step}" y="248" text-anchor="middle" font-size="12">{layer["layer"]}</text>')
    parts.append('<text x="335" y="267" text-anchor="middle" font-size="12">Layer</text>')
    for i,choice in enumerate(CHOICES):
        x = 48+i*150
        parts.append(f'<rect x="{x}" y="286" width="12" height="12" fill="{colors[choice]}"/>'
                     f'<text x="{x+18}" y="297" font-size="12">{labels[choice]}</text>')
    return ''.join(parts)+'</svg>'


def report_seed(parts, folder, item, position):
    seed = item.get('experiment_group', {}).get('seed', position)
    parts.append(f'<details class="chapter seed" id="seed-{h(seed)}"><summary>{position}. Analisi del seed {h(seed)}</summary><div class="chapter-body">')
    parts.append(f'<p class="muted">Checkpoint: epoca {h(item.get("checkpoint",{}).get("best_epoch","N/D"))}. '
                 'Ogni confronto è con l’esperto del medesimo task.</p>')
    for task in TASKS:
        result = item['splits'][f'{task}_test']
        expert = item['fixed_expert_test'][task]
        rm, em = report_metrics(result,task), report_metrics(expert,task)
        rc, ec = report_cost(result), report_cost(expert)
        saving = report_saving(ec,rc)
        parts.append(f'<section class="task" data-task="{task}"><h3>{task.upper()}</h3>')
        parts.append('<p class="answer"><strong>Come va?</strong> '+h(report_quality_sentence(100*(rm['f1']-em['f1'])))
                     + f' Risparmio di calcolo rispetto all’esperto: <strong>{fmt(saving,2)}%</strong>.</p>')
        parts.append(f'<h4>Qualità sul test — {rm["n"]} esempi</h4>')
        rows = [[label,fmt(100*em[key]),fmt(100*rm[key]),fmt(100*(rm[key]-em[key]),signed=True)]
                for key,label in [('accuracy','Accuracy'),('precision','Precision'),('recall','Recall'),('f1','F1')]]
        parts.append(table(['Metrica','Esperto '+task.upper()+' (%)','Routed (%)','Routed − esperto (punti)'],rows))
        parts.append('<p class="muted">'+report_metric_note(task)+' Differenze osservate; il bootstrap è nella sezione complessiva.</p>')
        parts.append('<h4>Risparmio in inferenza</h4>')
        rt, et = result.get('latency',{}).get('mean_ms_per_example'), expert.get('latency',{}).get('mean_ms_per_example')
        parts.append(table(['Modello','GFLOPs per esempio ↓','Tempo medio, ms per esempio ↓'],[
            ['Esperto '+task.upper(),fmt(ec/1e9 if ec else None),fmt(et)],
            ['Routed',fmt(rc/1e9 if rc else None),fmt(rt)]]))
        parts.append(f'<p><strong>Calcolo risparmiato: {fmt(saving,2)}%.</strong> '
                     + (f'Tempo risparmiato: {fmt(report_saving(et,rt),2)}%.' if rt is not None and et else 'Tempo non disponibile.')
                     + ' Nei FLOPs del routed è incluso il router.</p>')
        parts.append('<h4>Quali layer usa e quali salta?</h4><p>Ogni barra rappresenta un layer; '
                     'i colori indicano la percentuale di esempi instradata a ciascuna scelta sulla <strong>validation</strong>.</p>')
        record = item['splits'][f'{task}_validation']
        parts.append(report_routing_chart(record,task,seed))
        layers = record['routing']['per_layer']
        skipped = sum(row['hard_choice_percentages']['IDENTITY']/100 for row in layers)
        parts.append(f'<p>In media salta <strong>{fmt(skipped,2)} layer su {len(layers)}</strong> per esempio sulla validation. '
                     'Il numero di layer saltati è distinto dal risparmio FLOPs misurato sul test.</p>')
        parts.append('<details class="technical"><summary>Leggi le percentuali per layer</summary>'
                     +table(['Layer','Esperto SST %','Esperto MNLI %','Merged %','Salto %'],[
                         [row['layer'],*[fmt(row['hard_choice_percentages'][c],2) for c in CHOICES]] for row in layers])+'</details></section>')
    parts.append('</div></details>')


def report_aggregate(parts):
    parts.append(f'<details class="chapter aggregate" id="insieme" open><summary>{len(GROUP_RUNS)+1}. Come va il modello nel complesso?</summary><div class="chapter-body">')
    parts.append(f'<p>Media dei <strong>{len(GROUP_RUNS)} seed di addestramento completati</strong>. '
                 'Il valore dopo ± è la deviazione standard tra seed: indica quanto cambiano i risultati fra le esecuzioni. '
                 'L’esperto è fisso, comune ai seed: non è una media di esperti riaddestrati.</p>')
    for task in TASKS:
        results = [item['splits'][f'{task}_test'] for _,item in GROUP_RUNS]
        metrics = [report_metrics(r,task) for r in results]
        expert = GROUP_RUNS[0][1]['fixed_expert_test'][task]
        em, ec = report_metrics(expert,task), report_cost(expert)
        f1 = statistics.mean(m['f1'] for m in metrics)
        costs = [report_cost(r) for r in results]
        savings = [report_saving(ec,c) for c in costs]
        valid_savings = [s for s in savings if s is not None]
        parts.append(f'<section class="task" data-task="{task}"><h3>{task.upper()}</h3>')
        parts.append('<p class="answer"><strong>Risposta complessiva:</strong> '+h(report_quality_sentence(100*(f1-em['f1'])))
                     +f' Risparmio medio di calcolo: <strong>{fmt(statistics.mean(valid_savings) if valid_savings else None,2)}%</strong>.</p>')
        rows = [[label,fmt(100*em[key]),report_mean_std([m[key] for m in metrics],100),
                 fmt(100*(statistics.mean(m[key] for m in metrics)-em[key]),signed=True)]
                for key,label in [('accuracy','Accuracy'),('precision','Precision'),('recall','Recall'),('f1','F1')]]
        parts.append(table(['Metrica','Esperto fisso (%)','Routed: media ± DS (%)','Scarto medio rispetto all’esperto (punti)'],rows))
        parts.append('<p class="muted">'+report_metric_note(task)+'</p>')
        parts.append(table(['Misura di inferenza','Esperto fisso','Routed: media ± DS tra seed'],[
            ['GFLOPs per esempio',fmt(ec/1e9 if ec else None),report_mean_std(costs,1e-9)],
            ['Risparmio FLOPs rispetto all’esperto (%)','0',report_mean_std(savings)],
            ['Tempo, ms per esempio',fmt(expert.get('latency',{}).get('mean_ms_per_example')),
             report_mean_std([r.get('latency',{}).get('mean_ms_per_example') for r in results])]]))
        parts.append('<h4>Bootstrap: la differenza rispetto all’esperto è distinguibile dal caso?</h4>')
        boot_rows=[]
        for (_,item),result in zip(GROUP_RUNS,results):
            seed=item.get('experiment_group',{}).get('seed','N/D')
            boot=result.get('bootstrap_vs_expert')
            if not boot:
                boot_rows.append([seed,'N/D','N/D','N/D','Bootstrap non disponibile'])
                continue
            p=boot.get('p_value_two_sided')
            verdict='Non emerge una differenza al livello del 5%'
            if p is not None and p<.05:
                verdict='F1 routed inferiore all’esperto' if boot['delta_f1']<0 else 'F1 routed superiore all’esperto'
            if p is None: verdict='p-value non disponibile'
            boot_rows.append([seed,fmt(100*boot['delta_f1'],signed=True),
                              f"[{fmt(100*boot['ci95_low'])}; {fmt(100*boot['ci95_high'])}]",fmt(p,4),verdict])
        parts.append(table(['Seed','Routed − esperto: F1 (punti)','Intervallo di confidenza 95% (punti)','p-value','Lettura del test'],boot_rows))
        parts.append('<p class="muted">Bootstrap appaiato sugli stessi esempi. Ogni riga è il test già calcolato per quel seed, '
                     'non un test sulla media dei seed. I test sono letti separatamente al livello nominale del 5%. '
                     'Non rilevare una differenza non dimostra equivalenza.</p></section>')
    parts.append('</div></details>')


def report_baseline_label(candidate, name):
    policy=candidate['policy']
    method=policy['method']
    if method=='static':
        return f"Layer dropping statico — {len(policy['keep'])} layer mantenuti", 'Stessi layer dell’esperto del task mantenuti per tutti gli esempi. Una valutazione, senza riaddestramento.'
    if method=='random_expert_routing':
        return 'Routing casuale tra esperti', 'Per ogni esempio e layer sceglie casualmente fra SST, MNLI e salto. Media di tre semi casuali; non sono i tre seed di addestramento del routed.'
    return 'Early exit', 'L’esperto del task termina prima quando supera la soglia di confidenza. Riusa la testa finale ai layer intermedi. Una valutazione, senza riaddestramento.'


def report_baselines(parts):
    parts.append('<section id="baseline"><h2>Confronti con baseline a budget comparabile</h2>'
                 '<p><strong>Domanda:</strong> una tecnica più semplice conserva la stessa qualità del routed con circa lo stesso calcolo?</p>'
                 '<p>Il riferimento routed è la media dei seed di addestramento. Le baseline sono regolate sul suo costo medio di validation '
                 'e valutate sugli stessi input di test. Il costo effettivo è riportato in ogni confronto: non sempre coincide esattamente.</p>')
    for message in QC_MESSAGES:
        parts.append('<p class="notice">'+h(message)+'</p>')
    for task in TASKS:
        parts.append(f'<section class="task" data-task="{task}"><h3>{task.upper()}</h3>')
        rows=[r for r in QC_COMPARISONS if r['task']==task]
        if not rows:
            parts.append('<p>Confronti non disponibili per questo task. Non significa che il routed abbia vinto o perso.</p></section>')
            continue
        for row in rows:
            routed=row['routed']; rc=routed['inference_cost']['flops_per_example']; rf=routed['metrics']['f1']
            for name,candidate in row['baselines'].items():
                if candidate is None:
                    parts.append('<p>'+h(name)+': misura non disponibile.</p>')
                    continue
                title,description=report_baseline_label(candidate,name)
                result=candidate['test']; bc=result['inference_cost']['flops_per_example']; bf=result['metrics']['f1']
                delta=100*(rf-bf); cost_delta=100*(rc/bc-1)
                routed_text=fmt(100*rf)
                if routed.get('f1_std_seeds') is not None: routed_text+=' ± '+fmt(100*routed['f1_std_seeds'])
                baseline_text=fmt(100*bf)
                if result.get('f1_std_random') is not None: baseline_text+=' ± '+fmt(100*result['f1_std_random'])
                parts.append('<article class="comparison"><h4>'+h(title)+'</h4><p>'+h(description)+'</p>')
                parts.append(table(['Modello / tecnica','F1 sul test (%) ↑','GFLOPs per esempio ↓'],[
                    ['Routed — media dei seed di addestramento',routed_text,fmt(rc/1e9)],
                    [title+(' — media dei semi casuali' if 'f1_std_random' in result else ' — una valutazione'),baseline_text,fmt(bc/1e9)]]))
                quality='F1 maggiore' if delta>.0005 else 'F1 minore' if delta<-.0005 else 'F1 uguale alla precisione mostrata'
                cost='meno calcolo' if cost_delta<-.005 else 'più calcolo' if cost_delta>.005 else 'calcolo praticamente uguale'
                parts.append('<p class="answer"><strong>Il routed fa meglio o peggio?</strong> '
                             +h(f'{quality}: scarto routed − baseline {fmt(delta,signed=True)} punti F1. ')
                             +h(f'Usa {cost}: scarto {fmt(cost_delta,2,signed=True)}% rispetto al costo della baseline.')+'</p>')
                if delta>0 and cost_delta<0:
                    parts.append('<p>In questo confronto il routed ha qualità maggiore e costo minore.</p>')
                elif delta<0 and cost_delta>0:
                    parts.append('<p>In questo confronto la baseline ha qualità maggiore e costo minore.</p>')
                elif delta<0 and cost_delta<0:
                    parts.append('<p>Il routed risparmia calcolo, ma perde qualità rispetto a questa baseline.</p>')
                elif delta>0 and cost_delta>0:
                    parts.append('<p>Il routed migliora la qualità, ma usa più calcolo di questa baseline.</p>')
                parts.append('</article>')
        parts.append('</section>')
    parts.append('<p class="muted">Questi sono confronti dei valori osservati, non nuovi test di significatività. '
                 'Il bootstrap precedente confronta il routed con l’esperto completo. Il routing casuale usa gli esperti originali: '
                 'non è un confronto con il solo router sostituito e tutti gli altri pesi identici.</p></section>')


def report_curve_svg(groups, task):
    import io
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig,ax=plt.subplots(figsize=(9,4.8))
    for index,p in enumerate(groups,1):
        ax.errorbar(p['cost']/1e9,100*p['f1'],xerr=(p['cost_std'] or 0)/1e9,
                    yerr=100*(p['f1_std'] or 0),fmt='o',capsize=4,color='#2563eb')
        ax.annotate(f'C{index}',(p['cost']/1e9,100*p['f1']),xytext=(9,9),textcoords='offset points')
    if len(groups)>1:
        front=qc_pareto(groups)
        ax.scatter([p['cost']/1e9 for p in front],[100*p['f1'] for p in front],s=160,
                   facecolors='none',edgecolors='#dc2626',label='Configurazioni non dominate fra quelle misurate')
        ax.legend(fontsize=8,loc='best')
    ax.set(title=f'{task.upper()} — confronto tra configurazioni routed',xlabel='GFLOPs per esempio — meno è meglio',
           ylabel='F1 media sul test (%) — più è meglio')
    ax.grid(alpha=.2);fig.tight_layout()
    buf=io.StringIO();fig.savefig(buf,format='svg');plt.close(fig)
    return buf.getvalue()[buf.getvalue().index('<svg'):]


def report_curves(parts):
    parts.append('<section id="curve"><h2>Qualità e costo delle configurazioni routed</h2>'
                 '<p>Ogni punto è una configurazione routed; i suoi seed vengono aggregati. '
                 'Le barre mostrano la deviazione standard tra seed. Le baseline non compaiono in questi grafici.</p>')
    for task in TASKS:
        points=[p for p in QC_ARCHIVE['points'].values() if p['method']=='Routed' and p['task']==task
                and p['protocol_id']==QC_PROTOCOLS.get(task)]
        groups=sorted(qc_group_points(points),key=lambda p:p['config'])
        parts.append(f'<section class="task" data-task="{task}"><h3>{task.upper()}</h3>')
        if not groups:
            parts.append('<p>Nessuna configurazione routed confrontabile archiviata.</p></section>')
            continue
        if len(groups)==1:
            parts.append('<p class="notice"><strong>Una sola configurazione disponibile.</strong> '
                         'Il punto descrive la configurazione attuale; non è ancora possibile confrontare configurazioni diverse. '
                         'Le successive analisi aggiungeranno i nuovi punti all’archivio.</p>')
        parts.append(report_curve_svg(groups,task))
        parts.append(table(['Punto','Configurazione','Seed disponibili','F1 media ± DS (%)','GFLOPs medi ± DS'],[
            [f'C{i}',p['config'][:10],p['n_models'],fmt(100*p['f1'])+(' ± '+fmt(100*p['f1_std']) if p['f1_std'] is not None else ''),
             fmt(p['cost']/1e9)+(' ± '+fmt(p['cost_std']/1e9) if p['cost_std'] is not None else '')] for i,p in enumerate(groups,1)]))
        parts.append('<p>In alto: qualità maggiore. A sinistra: costo minore. '
                     'Quando ci sono più configurazioni, il cerchio rosso identifica quelle per cui nessun altro punto '
                     'ha qualità almeno uguale e costo almeno altrettanto basso, con un miglioramento in almeno uno dei due.</p>')
        parts.append('<details class="technical"><summary>Quali parametri definiscono ciascuna configurazione?</summary>')
        for i,p in enumerate(groups,1):
            parts.append(f'<h4>C{i} — {p["config"][:10]}</h4>'+table(['Parametro','Valore'],[
                [key,json.dumps(value,ensure_ascii=False)] for key,value in p['settings'].get('parameters',{}).items()]))
        parts.append('</details></section>')
    parts.append('<p class="muted">Si confrontano solo configurazioni valutate sugli stessi esempi e con lo stesso protocollo di misura. '
                 'La frontiera riguarda i valori medi osservati; non è un test statistico.</p></section>')


parts=['<!doctype html><html lang="it"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">',
       '<title>RoutedModel — risultati per seed, confronto complessivo e baseline</title>',
       '''<style>
body{margin:0;background:#eef2f6;color:#172b41;font:16px/1.6 system-ui,Arial,sans-serif}main{max-width:1100px;margin:auto;padding:32px;background:#fff}
h1{line-height:1.2;font-size:30px}h2{margin-top:44px;font-size:25px}h3{font-size:22px;color:#174a77}h4{font-size:18px;margin:24px 0 8px}
p{max-width:100ch}.muted{font-size:14px;color:#526275}.chapter{border:1px solid #cbd5e1;border-radius:8px;margin:18px 0}.chapter>summary{font-size:20px;padding:18px;background:#f4f7fb}
summary{cursor:pointer;font-weight:650}.chapter-body{padding:0 24px 24px}.task{margin:24px 0 32px}.task+.task{border-top:2px solid #e2e8f0;padding-top:12px}
.answer{background:#eff6ff;border-left:4px solid #2563eb;padding:14px 18px}.notice{background:#fff7e6;padding:12px 16px;border-left:4px solid #c08215}
.comparison{border:1px solid #dce4ed;border-radius:8px;padding:0 20px 12px;margin:20px 0}.table-wrap{overflow:auto}table{border-collapse:collapse;width:100%;font-size:14px;margin:14px 0}
th,td{text-align:left;padding:11px 12px;border-bottom:1px solid #dce4ed;vertical-align:top}th{background:#f0f4f8}tbody tr:hover{background:#f8fafc}
svg{display:block;max-width:100%;height:auto}.routing-chart{width:720px}.technical{font-size:14px;margin:18px 0}code{overflow-wrap:anywhere}nav a{display:inline-block;margin:4px 18px 4px 0}
@media(max-width:650px){main{padding:16px}.chapter-body{padding:0 12px 12px}.comparison{padding:0 10px 10px}h1{font-size:26px}}
@media print{body{background:white}main{padding:0}.chapter-body{display:block}nav{display:none}tr,svg{break-inside:avoid}}
</style></head><body><main>''',
       '<h1>Come va il modello routed?</h1>',
       f'<p>Esperimento <strong>{h(GROUP_DIR.name)}</strong> · {len(GROUP_RUNS)} seed completati.</p>',
       '<p>Prima trovi la stessa analisi per ogni seed, poi il risultato complessivo, i confronti con tecniche più semplici '
       'e infine i grafici delle configurazioni routed. Le sezioni dei seed e il riepilogo si aprono e si chiudono con un clic.</p>',
       '<nav aria-label="Sezioni">'+''.join(f'<a href="#seed-{h(item.get("experiment_group",{}).get("seed",i))}">Seed {h(item.get("experiment_group",{}).get("seed",i))}</a>'
                                            for i,(_,item) in enumerate(GROUP_RUNS,1))+
       '<a href="#insieme">Risultato complessivo</a><a href="#baseline">Confronti con baseline</a><a href="#curve">Configurazioni routed</a></nav>']
if QC_FAILURE is not None:
    parts.append('<p class="notice"><strong>Analisi delle baseline incompleta:</strong> '+h(str(QC_FAILURE))+'. I risultati disponibili restano leggibili.</p>')
for position,(folder,item) in enumerate(GROUP_RUNS,1):
    report_seed(parts,folder,item,position)
report_aggregate(parts)
report_baselines(parts)
report_curves(parts)
parts.append('<details class="technical"><summary>Provenienza e convenzioni di lettura</summary>'
             '<p>Qualità e costi: test. Grafici di routing per layer: validation. Precision e recall sono ricavati dalle matrici '
             'di confusione già salvate, senza nuove inferenze. DS: deviazione standard campionaria, ddof=1. '
             'N/D: dato non disponibile. Le differenze di qualità sono espresse in punti percentuali.</p>'
             '<p>I tempi misurano il forward a input già tokenizzati e trasferiti sulla GPU, con warm-up e sincronizzazione CUDA. '
             'Non includono tokenizzazione, trasferimenti e scrittura dei file. FLOPs e tempo sono misure distinte.</p>'
             '<p>Risultati: <code>'+h(str(GROUP_DIR))+'</code><br>Archivio qualità/costo: <code>'+h(str(ARCHIVE_PATH))+'</code></p></details>')
parts.append('</main></body></html>')
OUTPUT_PATH.write_text('\n'.join(parts),encoding='utf-8')
print('Report aggiornato:',OUTPUT_PATH)
if QC_FAILURE is not None:
    raise RuntimeError(f'Report scritto, ma le misure delle baseline sono incomplete: {QC_FAILURE}') from QC_FAILURE
